# Lesson 5: Response Streaming

Port of **Response Streaming** (Claude course: Lesson 7) to the Gemini API.

Claude emits named events (`MessageStart`, `ContentBlockStart`, `ContentBlockDelta`, `MessageStop`, ...).
Gemini instead yields a **stream of `GenerateContentResponse` chunks** — each chunk carries the newly generated `parts`, and the final chunk carries `finish_reason` / `usage_metadata`.

Same goal: show text as it arrives instead of waiting 10–30s for the complete answer.

In [1]:
import itertools
import time

from dotenv import load_dotenv
from google import genai
from google.genai import types
from google.genai.errors import ServerError

load_dotenv(override=True)

client = genai.Client()
model = "gemini-3.6-flash"

In [2]:
def add_user_message(messages, text):
    messages.append({"role": "user", "parts": [{"text": text}]})


def add_assistant_message(messages, text):
    # Gemini calls the assistant role "model"
    messages.append({"role": "model", "parts": [{"text": text}]})

## 1. Raw streaming — inspect every chunk

Claude equivalent: `client.messages.create(..., stream=True)` then `for event in stream: print(event)`.

In Gemini you call `generate_content_stream()` and iterate over chunks.
We stop after 2 chunks so the output stays readable — remove the `break` to dump the whole stream.

In [3]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

stream = client.models.generate_content_stream(
    model=model,
    contents=messages,
    config=types.GenerateContentConfig(max_output_tokens=1000),
)

for i, chunk in enumerate(stream):
    print(f"--- chunk {i} ---")
    print(chunk.model_dump(exclude_none=True))
    if i == 1:
        break

--- chunk 0 ---
{'sdk_http_response': {'headers': {'content-type': 'text/event-stream', 'content-disposition': 'attachment', 'vary': 'Origin, X-Origin, Referer', 'transfer-encoding': 'chunked', 'date': 'Mon, 26 Oct 2026 13:00:31 GMT', 'server': 'scaffolding on HTTPServer2', 'x-xss-protection': '0', 'x-frame-options': 'SAMEORIGIN', 'x-content-type-options': 'nosniff', 'server-timing': 'gfet4t7; dur=8104', 'alt-svc': 'h3=":443"; ma=2592000,h3-29=":443"; ma=2592000'}}, 'candidates': [{'content': {'parts': [{'text': 'MirageSQL is an enterprise-'}], 'role': 'model'}, 'index': 0}], 'model_version': 'gemini-3.6-flash', 'response_id': '507faoqrNJy4694PkP-x-Q4', 'usage_metadata': {'candidates_token_count': 7, 'prompt_token_count': 11, 'prompt_tokens_details': [{'modality': <MediaModality.TEXT: 'TEXT'>, 'token_count': 11}], 'thoughts_token_count': 438, 'total_token_count': 456}}
--- chunk 1 ---
{'sdk_http_response': {'headers': {'content-type': 'text/event-stream', 'content-disposition': 'attach

## 2. Simplified text streaming

Claude's `client.messages.stream(...)` + `stream.text_stream` becomes a small generator over `generate_content_stream` that yields only `chunk.text`.

`start_stream_with_retry` retries a 503 only **before** the first chunk arrives, so we never duplicate text the user has already seen.

In [4]:
def start_stream_with_retry(**params):
    """Return a chunk iterator, retrying only if the request fails before any text arrives."""
    for attempt in range(5):
        try:
            stream = client.models.generate_content_stream(**params)
            first = next(stream, None)  # the HTTP request is actually sent here
            if first is None:
                return iter(())
            return itertools.chain([first], stream)
        except ServerError:
            if attempt == 4:
                raise
            time.sleep(2 ** attempt)


def stream_text(messages, system=None, temperature=1.0):
    config_kwargs = {
        "max_output_tokens": 1000,
        "temperature": temperature,
    }
    if system:
        config_kwargs["system_instruction"] = system

    chunks = start_stream_with_retry(
        model=model,
        contents=messages,
        config=types.GenerateContentConfig(**config_kwargs),
    )
    for chunk in chunks:
        if chunk.text:
            yield chunk.text

In [5]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

start = time.perf_counter()
first_chunk_at = None
full_text = ""

for piece in stream_text(messages):
    if first_chunk_at is None:
        first_chunk_at = time.perf_counter()
    full_text += piece
    print(piece, end="", flush=True)

finished = time.perf_counter()
print()
print("no chunks received" if first_chunk_at is None else f"time to first chunk: {first_chunk_at - start:.2f}s")
print(f"total stream time:   {finished - start:.2f}s")
print(f"characters:          {len(full_text)}")

**AetherData** is a clandestine, quantum-relational database that continuously archives the unuttered thoughts, deleted texts, and forgotten dreams of every sentient being in the galaxy.
time to first chunk: 30.08s
total stream time:   30.21s
characters:          186


## 3. Getting the complete message

In Claude you call `stream.get_final_message()`. Gemini has no assembled message object on the stream, so we accumulate the chunks ourselves and keep the last `finish_reason` / `usage_metadata`.

This gives the same best-of-both-worlds: live text for the user, a complete record for storage.

In [6]:
def stream_and_collect(messages, system=None, temperature=1.0):
    """Stream text to the user, then return the assembled message for storage."""
    config_kwargs = {
        "max_output_tokens": 1000,
        "temperature": temperature,
    }
    if system:
        config_kwargs["system_instruction"] = system

    chunks = start_stream_with_retry(
        model=model,
        contents=messages,
        config=types.GenerateContentConfig(**config_kwargs),
    )

    parts = []
    finish_reason = None
    usage = None

    for chunk in chunks:
        if chunk.text:
            parts.append(chunk.text)
            print(chunk.text, end="", flush=True)
        if chunk.candidates and chunk.candidates[0].finish_reason:
            finish_reason = chunk.candidates[0].finish_reason
        if chunk.usage_metadata:
            usage = chunk.usage_metadata

    print()
    return {
        "role": "model",
        "text": "".join(parts),
        "finish_reason": finish_reason,
        "usage": usage,
    }


messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

final_message = stream_and_collect(messages)
add_assistant_message(messages, final_message["text"])

print("finish_reason:", final_message["finish_reason"])
if final_message["usage"]:
    print("prompt tokens:", final_message["usage"].prompt_token_count)
    print("output tokens:", final_message["usage"].candidates_token_count)
print("history:", [m["role"] for m in messages])

**VoidSQL** is a revolutionary, zero-latency database that achieves record-breaking query speeds by instantly discarding all incoming data and returning a randomized sense of closure.
finish_reason: FinishReason.STOP
prompt tokens: 11
output tokens: 33
history: ['user', 'model']


## 4. Streaming on the native chat session

The stateful `client.chats.create` object has `send_message_stream`, which keeps history for you across turns.

In [7]:
chat_session = client.chats.create(
    model=model,
    config=types.GenerateContentConfig(max_output_tokens=1000),
)

print("Turn 1: ", end="")
for chunk in chat_session.send_message_stream(
    "Write a 1 sentence description of a fake database"
):
    if chunk.text:
        print(chunk.text, end="", flush=True)
print("\n")

print("Turn 2: ", end="")
for chunk in chat_session.send_message_stream(
    "Now make it sound like a 1920s newspaper ad."
):
    if chunk.text:
        print(chunk.text, end="", flush=True)
print()

Turn 1: **ChronoSQL** is a temporal, multi-dimensional database designed to index data across parallel universes, allowing developers to query alternate realities and roll back production bugs into timelines where they never happened.

Turn 2: **MARVEL OF THE MODERN AGE!** Banish costly ledger blunders forever with **Messrs. Chrono & SQL’s Electro-Etheric Tabulator**, a patented mechanical wonder that siphons flawless business records directly from parallel dimensions, ensuring your firm’s accounts remain perpetually immaculate!
